# Real Data Pipeline - Analytics/Gold Layer
## Business-ready analytics and insights


In [0]:
# Purpose: Create business-ready analytics tables
 
import pyspark.sql.functions as F
from pyspark.sql.window import Window
 
catalog = "workspace"
schema = "github_analytics"
 
print("=" * 70)
print("GITHUB PIPELINE - ANALYTICS LAYER")
print("=" * 70)

In [0]:
# Cell 1: Load Silver Tables
silver_repos = spark.table(f"{catalog}.{schema}.silver_repositories")
silver_contribs = spark.table(f"{catalog}.{schema}.silver_contributors")
 
print("✅ Loaded silver tables")
 

In [0]:
# Cell 2: Gold Table 1 - Repository Rankings
print("\n" + "=" * 70)
print("CREATING GOLD TABLES")
print("=" * 70)

gold_repo_rankings = silver_repos \
    .select(
        "repo_name", "owner", "repo", "stars", "forks", "watchers",
        "language", "is_active", "days_since_update", "popularity_score"
    ) \
    .withColumn(
        "overall_rank",
        F.row_number().over(Window.orderBy(F.col("stars").desc()))
    ) \
    .withColumn(
        "language_rank",
        F.row_number().over(
            Window.partitionBy("language").orderBy(F.col("stars").desc())
        )
    ) \
    .withColumn(
        "tier",
        F.when(F.col("stars") >= 10000, "platinum")
         .when(F.col("stars") >= 5000, "gold")
         .when(F.col("stars") >= 1000, "silver")
         .otherwise("bronze")
    )

gold_table1 = f"{catalog}.{schema}.gold_repository_rankings"
gold_repo_rankings.write.format("delta").mode("overwrite").saveAsTable(gold_table1)

print(f"✅ Created: {gold_table1} ({gold_repo_rankings.count()} records)")


In [0]:
# Cell 3: Gold Table 2 - Contributor Analysis
gold_contributor_analysis = silver_repos.alias("r") \
    .join(
        silver_contribs.alias("c"),
        F.col("r.repo_name") == F.col("c.repo_name"),
        "inner"
    ) \
    .groupBy("c.contributor_login") \
    .agg(
        F.count("r.repo_name").alias("repos_contributed"),
        F.sum("c.contributions").alias("total_contributions"),
        F.collect_list("r.repo_name").alias("repos_list"),
        F.avg("r.stars").alias("avg_repo_stars"),
        F.max("r.stars").alias("max_repo_stars")
    ) \
    .withColumn(
        "expertise_level",
        F.when(F.col("total_contributions") >= 500, "expert")
         .when(F.col("total_contributions") >= 100, "experienced")
         .otherwise("beginner")
    ) \
    .withColumn(
        "contributor_rank",
        F.row_number().over(Window.orderBy(F.col("total_contributions").desc()))
    )

gold_table2 = f"{catalog}.{schema}.gold_contributor_analysis"
gold_contributor_analysis.write.format("delta").mode("overwrite").saveAsTable(gold_table2)

print(f"✅ Created: {gold_table2} ({gold_contributor_analysis.count()} records)")


In [0]:
# Cell 4: Gold Table 3 - Repository Ecosystem Health
gold_ecosystem_health = silver_repos \
    .select(
        "repo_name",
        "owner",
        "stars",
        "forks",
        "open_issues",
        "is_active",
        "days_since_update"
    ) \
    .withColumn(
        "health_score",
        F.round(
            (F.col("stars") / 100) * 0.4 +
            (F.col("forks") / 100) * 0.3 +
            (F.when(F.col("is_active"), 10).otherwise(0)) * 0.2 +
            (F.when(F.col("open_issues") < 100, 10).otherwise(5)) * 0.1,
            2
        )
    ) \
    .withColumn(
        "health_status",
        F.when(F.col("health_score") >= 70, "excellent")
         .when(F.col("health_score") >= 50, "good")
         .when(F.col("health_score") >= 30, "fair")
         .otherwise("needs_attention")
    )

gold_table3 = f"{catalog}.{schema}.gold_ecosystem_health"
gold_ecosystem_health.write.format("delta").mode("overwrite").saveAsTable(gold_table3)

print(f"✅ Created: {gold_table3} ({gold_ecosystem_health.count()} records)")


In [0]:
# Cell 5: Gold Table 4 - Language Trends
gold_language_trends = silver_repos \
    .filter(F.col("language").isNotNull()) \
    .groupBy("language") \
    .agg(
        F.count("repo_name").alias("total_repos"),
        F.sum("stars").alias("total_stars"),
        F.sum("forks").alias("total_forks"),
        F.avg("popularity_score").alias("avg_popularity"),
        F.sum(F.when(F.col("is_active"), 1).otherwise(0)).alias("active_repos"),
        F.count(F.when(F.col("days_since_update") <= 7, 1)).alias("recently_updated")
    ) \
    .withColumn(
        "momentum",
        F.round(F.col("recently_updated") / F.col("total_repos") * 100, 2)
    ) \
    .withColumn(
        "language_tier",
        F.row_number().over(Window.orderBy(F.col("total_stars").desc()))
    )

gold_table4 = f"{catalog}.{schema}.gold_language_trends"
gold_language_trends.write.format("delta").mode("overwrite").saveAsTable(gold_table4)

print(f"✅ Created: {gold_table4}")


In [0]:
# Cell 6: Gold Table 5 - Repository Comparison Matrix
gold_comparison_matrix = silver_repos \
    .select(
        "repo_name",
        "owner",
        "stars",
        "forks",
        "watchers",
        "open_issues",
        "language",
        "days_since_update",
        "popularity_score"
    ) \
    .withColumn(
        "stars_percentile",
        F.round(
            F.percent_rank().over(
                Window.orderBy(F.col("stars"))
            ) * 100,
            2
        )
    ) \
    .withColumn(
        "forks_percentile",
        F.round(
            F.percent_rank().over(
                Window.orderBy(F.col("forks"))
            ) * 100,
            2
        )
    )

gold_table5 = f"{catalog}.{schema}.gold_comparison_matrix"
gold_comparison_matrix.write.format("delta").mode("overwrite").saveAsTable(gold_table5)

print(f"✅ Created: {gold_table5} ({gold_comparison_matrix.count()} records)")


In [0]:

# Cell 7: Summary & Display Results
print("\n" + "=" * 70)
print("ANALYTICS LAYER COMPLETE")
print("=" * 70)

print("\n📊 TOP 10 REPOSITORIES BY STARS:")
display(gold_repo_rankings.orderBy(F.col("stars").desc()).limit(10))

print("\n🏆 TOP 10 CONTRIBUTORS:")
display(gold_contributor_analysis.orderBy(F.col("total_contributions").desc()).limit(10))

print("\n💪 ECOSYSTEM HEALTH STATUS:")
display(gold_ecosystem_health.groupBy("health_status").count())

print("\n📈 LANGUAGE TRENDS:")
display(gold_language_trends.orderBy(F.col("total_stars").desc()))